# LeetCode 1798 — Maximum Number of Consecutive Values You Can Make bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-consecutive-values-you-can-make/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def getMaximumConsecutive(self, coins: List[int]) -> int:
        ...
```

Lộ trình: **liệt kê mọi tập con** (`2ⁿ`) → **quy hoạch động tổng tập con** (tập/bitset) → **tham lam sau khi sắp xếp** `O(n log n)` (lời giải nộp) → **tham lam + counting sort** `O(n + V)`. Mấu chốt: nếu đang tạo được mọi giá trị `0..R-1` và có đồng xu `v ≤ R` thì tạo được `0..R+v-1`.

## 1. Mục tiêu học

1. Hiểu khái niệm **tổng tập con** (subset sum) và cách liệt kê / DP để biết những giá trị nào tạo được.
2. Nắm **bất biến "đoạn liên tục `[0, R)`"** và quy tắc mở rộng: đồng xu `v ≤ R` biến đoạn thành `[0, R + v)`.
3. Hiểu vì sao phải **sắp xếp tăng dần** trước khi tham lam, và vì sao gặp `v > R` thì **dừng hẳn** (`break`).
4. Biết dùng **số nguyên Python như bitset** (`mask | (mask << v)`) để DP tổng tập con nhanh cho input vừa.
5. Kiểm chứng tham lam bằng vét cạn ngẫu nhiên có seed và nhận ra bài "anh em" *Patching Array* (330) / *Minimum Number of Coins to be Added* (2952).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `coins` độ dài `n` — `n` đồng xu bạn đang có, đồng thứ `i` có giá trị `coins[i]`. Bạn **tạo được** giá trị `x` nếu chọn được một số đồng xu (mỗi đồng dùng tối đa một lần) có tổng đúng bằng `x`.

Trả về **số lượng lớn nhất các giá trị nguyên liên tiếp** bạn tạo được, **bắt đầu từ và bao gồm `0`**. Có thể có nhiều đồng xu cùng giá trị.

**Ví dụ 1:** `coins = [1,3]` → `2` (tạo được 0 bằng `[]`, 1 bằng `[1]`; không tạo được 2).

**Ví dụ 2:** `coins = [1,1,1,4]` → `8` (0..7 đều tạo được: 0, 1, 1+1, 1+1+1, 4, 4+1, 4+1+1, 4+1+1+1).

**Ví dụ 3:** `coins = [1,4,10,3,1]` → `20`.

**Ràng buộc:** `1 ≤ n ≤ 4·10⁴`, `1 ≤ coins[i] ≤ 4·10⁴`.

In [1]:
import random
import time
from itertools import combinations
from typing import List

examples = [([1, 3], 2), ([1, 1, 1, 4], 8), ([1, 4, 10, 3, 1], 20)]
for coins, want in examples:
    print(coins, "-> kỳ vọng", want)

[1, 3] -> kỳ vọng 2
[1, 1, 1, 4] -> kỳ vọng 8
[1, 4, 10, 3, 1] -> kỳ vọng 20


## 3. Kiến thức nền

### 3.1 Tổng tập con

Mỗi đồng xu "chọn hoặc không" ⇒ `2ⁿ` tập con. Tập con rỗng có tổng 0 — vì vậy 0 **luôn** tạo được.

In [2]:
coins = [1, 3]
for k in range(len(coins) + 1):
    for comb in combinations(coins, k):
        print(comb, "-> tổng", sum(comb))

() -> tổng 0
(1,) -> tổng 1
(3,) -> tổng 3
(1, 3) -> tổng 4


### 3.2 Đếm số giá trị liên tiếp từ 0

Cho tập `S` các tổng tạo được, đáp án là số `x` nhỏ nhất **không** thuộc `S` (gọi là *MEX* — minimum excluded). Vì `0..x-1` đều thuộc `S`, có đúng `x` giá trị liên tiếp.

In [3]:
def mex(S):
    x = 0
    while x in S:
        x += 1
    return x

print(mex({0, 1, 3, 4}), mex({0, 1, 2, 3, 4, 5, 6, 7}), mex({1, 2}))

2 8 0


### 3.3 Số nguyên Python làm bitset

Bit thứ `k` của `mask` bằng 1 nghĩa là "tạo được tổng `k`". Thêm đồng xu `v`: mọi tổng cũ `k` sinh thêm `k + v` ⇒ `mask |= mask << v`. Phép dịch/OR trên số nguyên lớn chạy trên cả khối bit cùng lúc nên nhanh.

In [4]:
mask = 1                       # chỉ tạo được tổng 0
for v in [1, 1, 1, 4]:
    mask |= mask << v
    print(f"thêm {v}: mask = {bin(mask)[2:][::-1]} (bit k từ trái sang = tổng k)")

thêm 1: mask = 11 (bit k từ trái sang = tổng k)
thêm 1: mask = 111 (bit k từ trái sang = tổng k)
thêm 1: mask = 1111 (bit k từ trái sang = tổng k)
thêm 4: mask = 11111111 (bit k từ trái sang = tổng k)


### 3.4 Đếm bit 1 liên tiếp từ bit 0

Số bit 1 liên tiếp tính từ bit thấp nhất của `mask` = vị trí của bit 0 đầu tiên. Công thức gọn: `(~mask & (mask + 1)).bit_length() - 1`. Phép `mask + 1` biến dãy 1 cuối thành 0 và bật bit 0 đầu tiên.

In [5]:
def trailing_ones(mask):
    return ((mask + 1) & ~mask).bit_length() - 1

for m in [0b0, 0b1, 0b1011, 0b11111111]:
    print(bin(m), "->", trailing_ones(m))

0b0 -> 0
0b1 -> 1
0b1011 -> 2
0b11111111 -> 8


### 3.5 Sắp xếp và `break`

`sorted(coins)` trả list tăng dần, `O(n log n)`. `break` thoát vòng lặp ngay — khác `continue` (bỏ phần còn lại của **lượt** này rồi lặp tiếp).

In [6]:
for v in sorted([4, 1, 10, 3, 1]):
    if v > 5:
        print("gặp", v, "-> break")
        break
    print("xử lý", v)

xử lý 1
xử lý 1
xử lý 3
xử lý 4
gặp 10 -> break


## 4. Ý tưởng nền tảng / trực giác

Đặt `R` = số giá trị liên tiếp đang tạo được, tức tạo được **mọi** giá trị `0, 1, …, R-1`. Ban đầu (chưa dùng đồng nào) `R = 1` (chỉ tạo được 0).

Thêm một đồng xu `v`:
- Các giá trị cũ vẫn tạo được: `[0, R)`.
- Cộng `v` vào mỗi giá trị cũ: `[v, v + R)`.
- *Nếu* `v ≤ R` thì hai đoạn **chạm/chồng** nhau ⇒ hợp là `[0, R + v)`.
- *Nếu* `v > R` thì giá trị `R` bị "hở": không tạo được bằng các đồng cũ, cũng không bằng `v + (cũ)` vì `v > R`.

Tính tay Ví dụ 3 sau khi sắp xếp `[1,1,3,4,10]`:

| đồng xu | `R` trước | `v ≤ R`? | `R` sau | đoạn tạo được |
|---|---|---|---|---|
| 1 | 1 | ✔ | 2 | `[0,2)` |
| 1 | 2 | ✔ | 3 | `[0,3)` |
| 3 | 3 | ✔ | 6 | `[0,6)` |
| 4 | 6 | ✔ | 10 | `[0,10)` |
| 10 | 10 | ✔ | 20 | `[0,20)` |

Đáp án 20. ✔ Vì sao phải sắp xếp? Nếu gặp `10` khi `R` mới là `1`, ta tưởng là hở — nhưng các đồng nhỏ đến sau có thể lấp được chỗ hở.

In [7]:
R = 1
for v in sorted([1, 4, 10, 3, 1]):
    old = R
    if v > R:
        print(f"v={v} > R={R}: hở tại {R}, dừng"); break
    R += v
    print(f"v={v:>2}: [0,{old}) ∪ [{v},{v+old}) = [0,{R})")
print("đáp án:", R)

v= 1: [0,1) ∪ [1,2) = [0,2)
v= 1: [0,2) ∪ [1,3) = [0,3)
v= 3: [0,3) ∪ [3,6) = [0,6)
v= 4: [0,6) ∪ [4,10) = [0,10)
v=10: [0,10) ∪ [10,20) = [0,20)
đáp án: 20


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi tập con

**Ý tưởng:** duyệt `2ⁿ` tập con, gom các tổng vào một `set`, rồi tính MEX.

In [8]:
def max_consecutive_brute(coins):
    sums = set()
    n = len(coins)
    for mask in range(1 << n):
        s = 0
        for i in range(n):
            if mask >> i & 1:
                s += coins[i]
        sums.add(s)
    return mex(sums)

for coins, want in examples:
    print(coins, "->", max_consecutive_brute(coins), "| kỳ vọng", want)

[1, 3] -> 2 | kỳ vọng 2
[1, 1, 1, 4] -> 8 | kỳ vọng 8
[1, 4, 10, 3, 1] -> 20 | kỳ vọng 20


**Độ phức tạp:** `O(2ⁿ · n)` thời gian, `O(2ⁿ)` bộ nhớ (worst số tổng khác nhau). `n = 20` đã là 20 triệu phép; `n = 4·10⁴` là vô vọng.

**Nút thắt:** nhiều tập con cho **cùng tổng** — ta liệt kê thừa. Chỉ cần biết **tổng nào** tạo được, không cần biết bằng tập con nào ⇒ DP trên tổng.

In [9]:
rng = random.Random(0)
c = [rng.randint(1, 30) for _ in range(16)]
t = time.perf_counter(); r = max_consecutive_brute(c); dt = time.perf_counter() - t
print(f"n=16: {r} trong {dt:.2f}s; mỗi đồng xu thêm vào nhân đôi thời gian")

n=16: 1 trong 0.04s; mỗi đồng xu thêm vào nhân đôi thời gian


## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến:** sau khi xét các đồng xu `c₁ ≤ c₂ ≤ … ≤ c_k` (đã sắp xếp), nếu chưa dừng thì tập tổng tạo được **chứa** đoạn `[0, R)` với `R = 1 + c₁ + … + c_k`, và vì tổng mọi đồng là `R − 1` nên tập đó **chính xác bằng** `[0, R)`.

**Quy tắc "nếu… thì…":**
- *Nếu* `c_{k+1} ≤ R` *thì* `R ← R + c_{k+1}` (đoạn mở rộng liền mạch).
- *Nếu* `c_{k+1} > R` *thì* giá trị `R` **không bao giờ** tạo được: các đồng đã xét có tổng tối đa `R − 1 < R`, còn bất kỳ tập con nào dùng một đồng chưa xét đều có tổng `≥ c_{k+1} > R`. ⇒ **dừng**, đáp án `R`.

Đây là lý do sắp xếp là bắt buộc: lập luận "mọi đồng chưa xét đều ≥ `c_{k+1}`" chỉ đúng khi duyệt tăng dần.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP tổng tập con bằng `set`, rồi bitset

Duy trì tập `S` các tổng tạo được; thêm đồng `v`: `S ← S ∪ {s + v | s ∈ S}`. Số tổng tối đa là `sum(coins) + 1`.

In [10]:
def max_consecutive_set(coins):
    S = {0}
    for v in coins:
        S |= {s + v for s in S}
    return mex(S)

def max_consecutive_bitset(coins):
    mask = 1
    for v in coins:
        mask |= mask << v
    return trailing_ones(mask)

for coins, want in examples:
    print(coins, max_consecutive_set(coins), max_consecutive_bitset(coins), want)

[1, 3] 2 2 2
[1, 1, 1, 4] 8 8 8
[1, 4, 10, 3, 1] 20 20 20


**Độ phức tạp:** gọi `S_tot = sum(coins)`. Bản `set`: `O(n · S_tot)` thời gian, `O(S_tot)` bộ nhớ. Bản bitset: cũng `O(n · S_tot / w)` với `w` là độ rộng từ máy (64) — nhanh hơn hàng chục lần. Nhưng `S_tot` có thể tới `4·10⁴ · 4·10⁴ = 1.6·10⁹` bit ≈ 200 MB **mỗi** lần dịch — vẫn không khả thi ở ràng buộc lớn nhất. Nút thắt: ta tính **toàn bộ** tập tổng, trong khi chỉ cần **đoạn đầu liên tục**.

### 7.2 Cách 3 — Tham lam sau khi sắp xếp (`O(n log n)`) — lời giải chuẩn

In [11]:
def max_consecutive_greedy_demo(coins):
    R = 1
    for v in sorted(coins):
        if v > R:
            print(f"  v={v} > R={R} -> dừng")
            break
        R += v
        print(f"  v={v}: R -> {R}")
    return R

for coins, want in examples:
    print(coins); print("  =>", max_consecutive_greedy_demo(coins), "| kỳ vọng", want)

[1, 3]
  v=1: R -> 2
  v=3 > R=2 -> dừng
  => 2 | kỳ vọng 2
[1, 1, 1, 4]
  v=1: R -> 2
  v=1: R -> 3
  v=1: R -> 4
  v=4: R -> 8
  => 8 | kỳ vọng 8
[1, 4, 10, 3, 1]
  v=1: R -> 2
  v=1: R -> 3
  v=3: R -> 6
  v=4: R -> 10
  v=10: R -> 20
  => 20 | kỳ vọng 20


In [12]:
class SolutionGreedy:
    def getMaximumConsecutive(self, coins: List[int]) -> int:
        R = 1
        for v in sorted(coins):
            if v > R:
                break
            R += v
        return R

for coins, want in examples:
    print(SolutionGreedy().getMaximumConsecutive(coins), want)

2 2
8 8
20 20


**Độ phức tạp:** sắp xếp `O(n log n)` + một lượt `O(n)` ⇒ `O(n log n)`; bộ nhớ `O(n)` cho mảng đã sắp xếp (hoặc `O(1)` phụ nếu `coins.sort()` tại chỗ).

### 7.3 Cách 4 — Tham lam + counting sort (`O(n + V)`)

Giá trị đồng xu ≤ `V = 4·10⁴`, nên có thể đếm số đồng mỗi giá trị rồi duyệt `v = 1..V`. Khi `v ≤ R`, thêm **tất cả** `cnt[v]` đồng cùng lúc: mỗi đồng `v` vẫn thỏa `v ≤ R` (vì `R` chỉ tăng), nên `R += v · cnt[v]`.

In [13]:
class SolutionCounting:
    def getMaximumConsecutive(self, coins: List[int]) -> int:
        V = max(coins)
        cnt = [0] * (V + 1)
        for v in coins:
            cnt[v] += 1
        R = 1
        for v in range(1, V + 1):
            if cnt[v] == 0:
                continue
            if v > R:
                break
            R += v * cnt[v]
        return R

for coins, want in examples:
    print(SolutionCounting().getMaximumConsecutive(coins), want)

2 2
8 8
20 20


**Độ phức tạp:** `O(n + V)` thời gian, `O(V)` bộ nhớ. Ở đây `continue` khi `cnt[v] == 0` là **đúng** (không có đồng giá trị đó thì chẳng có gì để xét), còn `break` khi `v > R` là bắt buộc.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 2 `coins = [1,1,1,4]` và trên một ví dụ dừng sớm `[1,3]`.

| bước | `v` | `R` trước | `v ≤ R`? | `R` sau | giá trị tạo được |
|---|---|---|---|---|---|
| 1 | 1 | 1 | ✔ | 2 | 0..1 |
| 2 | 1 | 2 | ✔ | 3 | 0..2 |
| 3 | 1 | 3 | ✔ | 4 | 0..3 |
| 4 | 4 | 4 | ✔ (bằng nhau vẫn được) | 8 | 0..7 |

Với `[1,3]`: sau đồng 1, `R = 2`; đồng 3 > 2 ⇒ giá trị 2 hở ⇒ đáp án 2.

Hàm trace dưới đây còn **in ra tập tổng thật** (bằng bitset) sau mỗi bước để đối chiếu bất biến:

In [14]:
def trace_greedy(coins):
    R = 1
    mask = 1
    for step, v in enumerate(sorted(coins), 1):
        if v > R:
            print(f"bước {step}: v={v} > R={R} -> dừng; giá trị {R} không tạo được")
            break
        R += v
        mask |= mask << v
        made = [k for k in range(mask.bit_length()) if mask >> k & 1]
        print(f"bước {step}: v={v}, R={R:>2} | tập tổng thật = {made} | khớp [0,R): {made == list(range(R))}")
    return R

trace_greedy([1, 1, 1, 4]); print()
trace_greedy([1, 3])

bước 1: v=1, R= 2 | tập tổng thật = [0, 1] | khớp [0,R): True
bước 2: v=1, R= 3 | tập tổng thật = [0, 1, 2] | khớp [0,R): True
bước 3: v=1, R= 4 | tập tổng thật = [0, 1, 2, 3] | khớp [0,R): True
bước 4: v=4, R= 8 | tập tổng thật = [0, 1, 2, 3, 4, 5, 6, 7] | khớp [0,R): True

bước 1: v=1, R= 2 | tập tổng thật = [0, 1] | khớp [0,R): True
bước 2: v=3 > R=2 -> dừng; giá trị 2 không tạo được


2

## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def getMaximumConsecutive(self, coins: List[int]) -> int:
        coins.sort()
        R = 1                      # đang tạo được mọi giá trị 0..R-1
        for v in coins:
            if v > R:              # giá trị R bị hở, đồng sau còn lớn hơn
                break
            R += v
        return R

for coins, want in examples:
    assert Solution().getMaximumConsecutive(list(coins)) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


(`coins.sort()` sắp xếp **tại chỗ**, làm thay đổi list đầu vào — LeetCode chấp nhận; khi test cục bộ ta truyền bản sao `list(coins)`.)

## 10. Vì sao thuật toán đúng

Gọi `c₁ ≤ … ≤ c_n` là các đồng xu đã sắp xếp, `R_k = 1 + c₁ + … + c_k`.

**Bổ đề (quy nạp):** nếu `c_i ≤ R_{i-1}` với mọi `i ≤ k` thì tập tổng của `{c₁..c_k}` là đúng `[0, R_k)`.
- `k = 0`: chỉ có tổng 0 = `[0, 1)`. ✔
- Bước: tập tổng mới = `[0, R_{k-1}) ∪ [c_k, c_k + R_{k-1})`. Do `c_k ≤ R_{k-1}`, hai đoạn liền nhau ⇒ hợp là `[0, R_{k-1} + c_k) = [0, R_k)`. ✔

**Dừng đúng:** gọi `k` là chỉ số đầu tiên với `c_k > R_{k-1}`. Xét giá trị `x = R_{k-1}`:
- Tập con chỉ gồm các đồng `c₁..c_{k-1}`: tổng tối đa `R_{k-1} − 1 < x`.
- Tập con có ít nhất một đồng `c_j` với `j ≥ k`: tổng `≥ c_j ≥ c_k > x`.

⇒ `x` không tạo được, và `0..x-1` tạo được ⇒ đáp án đúng bằng `x = R_{k-1}`. Nếu không dừng, đáp án `R_n = 1 + sum(coins)` — không thể tạo giá trị lớn hơn tổng mọi đồng.

## 11. Phân tích Time / Space

Gọi `n = len(coins)`, `V = max(coins) ≤ 4·10⁴`, `S = sum(coins) ≤ 1.6·10⁹`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê tập con | `O(2ⁿ · n)` | `O(2ⁿ)` |
| 2a. DP `set` | `O(n · S)` | `O(S)` |
| 2b. DP bitset | `O(n · S / 64)` | `O(S / 8)` byte |
| 3. Sắp xếp + tham lam | `O(n log n)` | `O(1)` phụ (sort tại chỗ) |
| 4. Counting sort + tham lam | `O(n + V)` | `O(V)` |

In [16]:
rng = random.Random(1798)
big = [rng.randint(1, 4 * 10**4) for _ in range(4 * 10**4)]
big_ones = [1] * (4 * 10**4)                 # đi hết mảng, không dừng sớm
for name, sol in [("Cách 3", SolutionGreedy()), ("Cách 4", SolutionCounting()), ("Nộp", Solution())]:
    for label, data in [("ngẫu nhiên", big), ("toàn 1", big_ones)]:
        t = time.perf_counter()
        r = sol.getMaximumConsecutive(list(data))
        print(f"{name} ({label}): {r} trong {1000*(time.perf_counter()-t):.1f} ms")
mid = [rng.randint(1, 300) for _ in range(300)]
t = time.perf_counter(); r = max_consecutive_bitset(mid); dt = time.perf_counter() - t
print(f"Bitset n=300, S={sum(mid)}: {r} trong {1000*dt:.1f} ms (đúng: {r == Solution().getMaximumConsecutive(list(mid))})")

Cách 3 (ngẫu nhiên): 1 trong 5.4 ms
Cách 3 (toàn 1): 40001 trong 1.0 ms
Cách 4 (ngẫu nhiên): 1 trong 1.3 ms
Cách 4 (toàn 1): 40001 trong 1.5 ms
Nộp (ngẫu nhiên): 1 trong 5.0 ms
Nộp (toàn 1): 40001 trong 1.0 ms
Bitset n=300, S=43815: 1 trong 0.3 ms (đúng: True)


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
sols = [Solution(), SolutionGreedy(), SolutionCounting()]
edge = examples + [
    ([1], 2),
    ([2], 1),                 # không có đồng 1 -> chỉ tạo được 0
    ([1, 1], 3),
    ([1, 2, 5], 4),
    ([1, 2, 4, 8], 16),       # lũy thừa 2: tạo được mọi số < 16
    ([5, 1, 1, 1, 1], 10),    # thứ tự đầu vào lộn xộn
    ([40000], 1),
    ([1] * 10, 11),
]
for coins, want in edge:
    for s in sols:
        assert s.getMaximumConsecutive(list(coins)) == want, (coins, s)
    assert max_consecutive_brute(coins) == want == max_consecutive_set(coins) == max_consecutive_bitset(coins)
    print("OK", coins, "->", want)

OK [1, 3] -> 2
OK [1, 1, 1, 4] -> 8
OK [1, 4, 10, 3, 1] -> 20
OK [1] -> 2
OK [2] -> 1
OK [1, 1] -> 3
OK [1, 2, 5] -> 4
OK [1, 2, 4, 8] -> 16
OK [5, 1, 1, 1, 1] -> 10
OK [40000] -> 1
OK [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] -> 11


### 12.2 Ngẫu nhiên có seed, đối chiếu vét cạn

In [18]:
rng = random.Random(48)
for _ in range(600):
    n = rng.randint(1, 11)
    hi = rng.choice([3, 6, 15, 40])
    coins = [rng.randint(1, hi) for _ in range(n)]
    want = max_consecutive_brute(coins)
    for s in sols:
        assert s.getMaximumConsecutive(list(coins)) == want, coins
    assert max_consecutive_bitset(coins) == want
for _ in range(50):
    coins = [rng.randint(1, 200) for _ in range(rng.randint(50, 200))]
    assert max_consecutive_bitset(coins) == Solution().getMaximumConsecutive(list(coins))
print("600 case vét cạn + 50 case bitset: khớp")

600 case vét cạn + 50 case bitset: khớp


### 12.3 Kiểm tra bất biến "tập tổng = [0, R)" ở mọi bước chưa dừng

In [19]:
rng = random.Random(6)
for _ in range(300):
    coins = sorted(rng.randint(1, 8) for _ in range(rng.randint(1, 10)))
    R, mask = 1, 1
    for v in coins:
        if v > R:
            assert not (mask >> R & 1)      # R thật sự không tạo được
            break
        R += v; mask |= mask << v
        assert mask == (1 << R) - 1         # tập tổng đúng bằng [0, R)
print("bất biến đúng trên 300 dãy")

bất biến đúng trên 300 dãy


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp

In [20]:
def wrong_no_sort(coins):
    R = 1
    for v in coins:                      # SAI: không sắp xếp
        if v > R:
            break
        R += v
    return R

coins = [3, 1]
print("sai:", wrong_no_sort(coins), "| đúng:", Solution().getMaximumConsecutive(list(coins)))

sai: 1 | đúng: 2


**Vì sao sai:** gặp `3` khi `R = 1` thì dừng, nhưng đồng `1` đến sau lấp được chỗ hở. Lập luận "mọi đồng sau đều lớn hơn" chỉ đúng khi đã sắp xếp.

### Lỗi 2 — Không sắp xếp nhưng "bỏ qua" đồng lớn bằng `continue`

In [21]:
def wrong_skip(coins):
    R = 1
    for v in coins:
        if v > R:
            continue                     # SAI: đồng bị bỏ qua không được xét lại
        R += v
    return R

coins = [3, 1, 1]
print("sai:", wrong_skip(coins), "| đúng:", Solution().getMaximumConsecutive(list(coins)))

sai: 3 | đúng: 6


**Vì sao sai:** sau hai đồng `1`, `R = 3` và đồng `3` **lẽ ra** dùng được (`3 ≤ 3`), nhưng nó đã bị bỏ qua từ trước. Kết quả 3 thay vì 6. **Sửa:** sắp xếp rồi `break`.

### Lỗi 3 — Nhầm `R` (số giá trị) với `X = R − 1` (giá trị lớn nhất) trong điều kiện

In [22]:
def wrong_condition(coins):
    X = 0                                # giá trị lớn nhất tạo được liên tục
    for v in sorted(coins):
        if v > X:                        # SAI: phải là v > X + 1
            break
        X += v
    return X + 1

for coins in ([1], [1, 1, 1, 4]):
    print(coins, "sai:", wrong_condition(coins), "| đúng:", Solution().getMaximumConsecutive(list(coins)))

[1] sai: 1 | đúng: 2
[1, 1, 1, 4] sai: 1 | đúng: 8


**Vì sao sai:** nếu tạo được `0..X` thì đồng `v` nối liền khi `v ≤ X + 1` (để `v + 0 = v` chạm ngay sau `X`). Viết `v > X` khiến đồng `1` đầu tiên (khi `X = 0`) bị từ chối. **Sửa:** dùng `R = X + 1` và điều kiện `v > R` như lời giải, hoặc `v > X + 1`.

### Lỗi 4 — Quên đếm giá trị 0

In [23]:
def wrong_forget_zero(coins):
    X = 0
    for v in sorted(coins):
        if v > X + 1:
            break
        X += v
    return X                             # SAI: đây là giá trị lớn nhất, không phải số lượng

print("sai:", wrong_forget_zero([1, 3]), "| đúng:", Solution().getMaximumConsecutive([1, 3]))

sai: 1 | đúng: 2


**Vì sao sai:** đề đếm **số lượng** giá trị liên tiếp **bao gồm 0**: `0..X` có `X + 1` giá trị. **Sửa:** trả `X + 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tạo được mọi giá trị từ 0/1 tới …" với đồng xu/số dùng một lần** ⇒ sắp xếp + duy trì `R` (đoạn liên tục `[0, R)`), quy tắc `v ≤ R`.
- Đây là "pattern **đoạn phủ liên tục**" — giống bài *Patching Array* (330), *Minimum Number of Coins to be Added* (2952), *Jump Game* (phủ bằng tầm với).
- DP tổng tập con chỉ cần khi hỏi **một giá trị cụ thể** hoặc **mọi** giá trị (không chỉ đoạn đầu).
- Bitset bằng số nguyên Python: `mask |= mask << v` — vũ khí tốt cho subset-sum cỡ vừa.
- Phân biệt "số lượng giá trị" (`R`) và "giá trị lớn nhất" (`R − 1`).

## 15. Biến thể của bài

1. **LeetCode 330 — Patching Array:** cho mảng đã sắp xếp và `n`, thêm ít số nhất để tạo mọi giá trị `1..n`: khi gặp hở `R`, thêm chính số `R` (gấp đôi đoạn).
2. **LeetCode 2952 — Minimum Number of Coins to be Added:** giống 330 với `target` (Bài tập 5).
3. **Mỗi đồng dùng vô hạn lần:** có đồng 1 thì tạo được mọi giá trị; bài hỏi giá trị lớn nhất không tạo được (*Frobenius*) khó hơn nhiều.
4. **Hỏi giá trị nhỏ nhất ≥ 1 không tạo được** = chính đáp án `R` của bài này (khi `R ≥ 1`).
5. **Hỏi có tạo được tổng `x` hay không** ⇒ subset-sum DP / bitset (Cách 2).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê tập con | `2ⁿ` tổng, MEX | `O(2ⁿ n)` | `O(2ⁿ)` | kiểm chứng `n ≤ 15` |
| 2. DP set/bitset | tập tổng tạo được | `O(n S)` (/64) | `O(S)` | khi cần **mọi** tổng, `S` vừa |
| 3. Sort + tham lam | đoạn `[0, R)`, `v ≤ R` | `O(n log n)` | `O(1)` phụ | **nộp bài** |
| 4. Counting + tham lam | như 3, đếm theo giá trị | `O(n + V)` | `O(V)` | giá trị nhỏ, `n` lớn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `getMaximumConsecutive([2])`, `([1, 2, 5])`, `([1, 2, 3, 10])`?

**Bài 2 (tính tay):** Lập bảng `v, R` cho `coins = [7, 1, 2, 1, 5]`.

**Bài 3 (điền TODO):** Viết `reachable_sums(coins)` trả về **list đã sắp xếp** mọi tổng tạo được, dùng bitset.

**Bài 4 (tự viết lại):** Viết lại tham lam **không sắp xếp toàn bộ** mà dùng **heap** (`heapq`) lấy dần đồng nhỏ nhất — dừng sớm khi hở (có lợi khi hở xuất hiện sớm).

**Bài 5 (mini challenge — LeetCode 2952):** Cho `coins` và `target`. Tìm **số đồng xu ít nhất** cần **thêm vào** (giá trị tùy chọn) để mọi giá trị trong `[1, target]` đều tạo được.

In [24]:
def reachable_sums(coins):
    # TODO: bitset
    pass

def max_consecutive_heap(coins):
    # TODO: heapq
    pass

def min_coins_to_add(coins, target):
    # TODO: tham lam "vá hở" bằng chính giá trị R
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Bắt đầu `R = 1` | `[2]`: 2 > 1 | `[1,2,3,10]`: R = 7 rồi 10 > 7 |
| 2 | Sắp xếp `[1,1,2,5,7]` | R: 1→2→3→5→10 | 7 ≤ 10 ⇒ 17 |
| 3 | `mask = 1`, `mask |= mask << v` | Duyệt bit | `[k for k in range(mask.bit_length()) if mask >> k & 1]` |
| 4 | `heapq.heapify(h)` `O(n)` | `heappop` lần lượt | Dừng khi `h[0] > R` |
| 5 | Duy trì `R` (đoạn `[0, R)`), dừng khi `R > target` | Nếu đồng tiếp theo `> R` (hoặc hết đồng) thì thêm đồng giá trị `R`: `R *= 2`, đếm +1 | Ngược lại `R += v` |

## 19. Đáp án tham khảo

In [25]:
print("Bài 1:", [Solution().getMaximumConsecutive(c) for c in ([2], [1, 2, 5], [1, 2, 3, 10])])
assert [Solution().getMaximumConsecutive(c) for c in ([2], [1, 2, 5], [1, 2, 3, 10])] == [1, 4, 7]
print("Bài 2:"); assert trace_greedy([7, 1, 2, 1, 5]) == 17

Bài 1: [1, 4, 7]
Bài 2:
bước 1: v=1, R= 2 | tập tổng thật = [0, 1] | khớp [0,R): True
bước 2: v=1, R= 3 | tập tổng thật = [0, 1, 2] | khớp [0,R): True
bước 3: v=2, R= 5 | tập tổng thật = [0, 1, 2, 3, 4] | khớp [0,R): True
bước 4: v=5, R=10 | tập tổng thật = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | khớp [0,R): True
bước 5: v=7, R=17 | tập tổng thật = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16] | khớp [0,R): True


In [26]:
import heapq

def reachable_sums(coins):
    mask = 1
    for v in coins:
        mask |= mask << v
    return [k for k in range(mask.bit_length()) if mask >> k & 1]

def max_consecutive_heap(coins):
    h = list(coins)
    heapq.heapify(h)
    R = 1
    while h and h[0] <= R:
        R += heapq.heappop(h)
    return R

rng = random.Random(17)
for _ in range(300):
    coins = [rng.randint(1, 12) for _ in range(rng.randint(1, 10))]
    rs = reachable_sums(coins)
    assert rs == sorted({sum(c) for k in range(len(coins) + 1) for c in combinations(coins, k)})
    assert max_consecutive_heap(coins) == max_consecutive_brute(coins)
print("Bài 3, 4: OK", reachable_sums([1, 3]))

Bài 3, 4: OK [0, 1, 3, 4]


In [27]:
def min_coins_to_add(coins, target):
    coins = sorted(coins)
    R, i, added = 1, 0, 0              # tạo được [0, R)
    while R <= target:
        if i < len(coins) and coins[i] <= R:
            R += coins[i]; i += 1
        else:
            R += R; added += 1         # thêm đồng giá trị R: [0, 2R)
    return added

def min_coins_to_add_brute(coins, target):
    # thử thêm k đồng (giá trị 1..target, có lặp) với k = 0, 1, 2, ...
    from itertools import combinations_with_replacement
    k = 0
    while True:
        for extra in combinations_with_replacement(range(1, target + 1), k):
            if max_consecutive_bitset(list(coins) + list(extra)) > target:
                return k
        k += 1

assert min_coins_to_add([1, 4, 10], 19) == 2       # ví dụ của LeetCode 2952
assert min_coins_to_add([1, 4, 10, 5, 7, 19], 19) == 1
assert min_coins_to_add([1, 1, 1], 20) == 3
rng = random.Random(29)
for _ in range(150):
    coins = [rng.randint(1, 10) for _ in range(rng.randint(1, 4))]
    target = rng.randint(1, 14)
    assert min_coins_to_add(coins, target) == min_coins_to_add_brute(coins, target), (coins, target)
print("Bài 5: OK (đối chiếu vét cạn)")

Bài 5: OK (đối chiếu vét cạn)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
coins.sort()
R = 1                       # tạo được mọi giá trị trong [0, R)
for v in coins:
    if v > R: break         # giá trị R bị hở, không thể lấp
    R += v                  # [0, R) ∪ [v, v+R) = [0, R+v)
return R
```

**Checklist:**
- [ ] Đã **sắp xếp tăng dần**?
- [ ] Khởi tạo `R = 1` (tính giá trị 0)?
- [ ] Điều kiện dừng `v > R` (bằng nhau vẫn nối được)?
- [ ] Dùng `break`, không `continue` (trên mảng đã sắp xếp `continue` cũng ra đúng nhưng lãng phí; trên mảng chưa sắp xếp thì sai)?
- [ ] Trả về **số lượng** `R`, không phải giá trị lớn nhất `R − 1`.